# Phase 3 — E-Commerce Customer Churn
## Notebook 02: XGBoost Modelling + SHAP Explainability


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    roc_auc_score, average_precision_score
)
from xgboost import XGBClassifier, plot_importance
import shap

# ── Portable paths ────────────────────────────────────────────────────────────
ROOT      = os.path.dirname(os.path.abspath('__file__'))
DATA_PROC = os.path.join(ROOT, '..', 'data', 'processed', 'churn_cleaned.csv')
FIG_DIR   = os.path.join(ROOT, '..', 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

%matplotlib inline

## 1. Load Cleaned Data

In [7]:
df = pd.read_csv(DATA_PROC)
print('Shape:', df.shape)
print('Churn dist:\n', df['Churn'].value_counts())
df.head()

Shape: (5630, 20)
Churn dist:
 Churn
0    4682
1     948
Name: count, dtype: int64


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.000000,Mobile Phone,3,6.0,Debit Card,Female,3.000000,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,10.189899,Phone,1,8.0,UPI,Male,3.000000,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,10.189899,Phone,1,30.0,Debit Card,Male,2.000000,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.000000,Phone,3,15.0,Debit Card,Male,2.000000,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.000000,Phone,1,12.0,CC,Male,2.931535,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


## 2. Feature Engineering & Encoding

In [8]:
categorical_feature = df.select_dtypes(include=['object']).columns.tolist()
numerical_feature   = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

if 'Churn' in numerical_feature:
    numerical_feature.remove('Churn')
if 'CustomerID' in numerical_feature:
    numerical_feature.remove('CustomerID')

df_encoded = pd.get_dummies(df, columns=categorical_feature, drop_first=False)
df_encoded = df_encoded.drop(columns=['CustomerID'], errors='ignore')

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print('Features after encoding:', X.shape[1])
print('Target distribution:\n', y.value_counts())

Features after encoding: 34
Target distribution:
 Churn
0    4682
1     948
Name: count, dtype: int64


/var/folders/4b/0w0rsx895vd1lgwp5z1c6g640000gn/T/ipykernel_83687/3088494759.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_feature = df.select_dtypes(include=['object']).columns.tolist()


## 3. Train / Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

Train: (4504, 34) | Test: (1126, 34)


## 4. XGBoost Model — Handling Class Imbalance

In [10]:
model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=3,
    learning_rate=0.1,
    eval_metric='logloss',
    reg_alpha=1.0,   # L1
    reg_lambda=0.0,  # L2
)

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    sample_weight=sample_weights,
    verbose=False
)
print('Model trained ✓')

NameError: name 'XGBClassifier' is not defined

## 5. Evaluation

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('Accuracy :', accuracy_score(y_test, y_pred))
print('ROC AUC  :', roc_auc_score(y_test, y_proba))
print('Avg Prec :', average_precision_score(y_test, y_proba))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test, normalize='true', cmap='Blues', ax=ax
)
ax.set_title('Normalized Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax)
ax.set_title('ROC Curve')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'roc_curve.png'), dpi=150)
plt.show()

# Precision-Recall Curve
fig, ax = plt.subplots(figsize=(7, 5))
PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=ax)
ax.set_title('Precision-Recall Curve')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'pr_curve.png'), dpi=150)
plt.show()

In [ ]:
# Feature Importance
fig, ax = plt.subplots(figsize=(10, 8))
plot_importance(model, max_num_features=20, importance_type='gain', ax=ax)
ax.set_title('Top 20 Feature Importances (Gain)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'feature_importance.png'), dpi=150)
plt.show()

## 6. SHAP Explainability

In [ ]:
X_train_shap = X_train.astype('float64')
X_test_shap  = X_test.astype('float64')

background = shap.sample(X_train_shap, 200, random_state=42)
explainer  = shap.TreeExplainer(model, data=background, model_output='probability')
shap_values = explainer(X_test_shap)

print('SHAP values computed ✓')

In [ ]:
# Beeswarm — global feature importance
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'shap_beeswarm.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Waterfall — highest-risk customer
y_proba = model.predict_proba(X_test_shap)[:, 1]
highest_risk_pos = np.argmax(y_proba)

print(f'Customer index : {X_test_shap.index[highest_risk_pos]}')
print(f'Churn prob     : {y_proba[highest_risk_pos]:.4f}')
print(f'Actual label   : {y_test.iloc[highest_risk_pos]}')

shap.plots.waterfall(shap_values[highest_risk_pos], max_display=15, show=False)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'shap_waterfall_highrisk.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Cost-Benefit Analysis

A common business frame: catching one churner saves the company ~$X in revenue.
A false positive costs ~$Y in wasted retention offer.

In [ ]:
# Adjust thresholds and compare profit
REVENUE_SAVED_PER_TP = 200   # $ saved per correctly caught churner
COST_PER_FP          = 20    # $ wasted per false positive (retention offer)

thresholds = np.arange(0.1, 0.9, 0.05)
profits = []

for t in thresholds:
    preds = (y_proba >= t).astype(int)
    tp = ((preds == 1) & (y_test == 1)).sum()
    fp = ((preds == 1) & (y_test == 0)).sum()
    profit = tp * REVENUE_SAVED_PER_TP - fp * COST_PER_FP
    profits.append(profit)

best_t = thresholds[np.argmax(profits)]
print(f'Best threshold: {best_t:.2f}  |  Max profit: ${max(profits):,.0f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, profits, marker='o', color='#4C72B0')
ax.axvline(best_t, color='red', linestyle='--', label=f'Best t={best_t:.2f}')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Estimated Profit ($)')
ax.set_title('Cost-Benefit Analysis — Churn Threshold Optimisation')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'cost_benefit_analysis.png'), dpi=150)
plt.show()

## 8. Business Conclusions

| Metric | Value |
|---|---|
| AUC-ROC | *(fill after real data run)* |
| Average Precision | *(fill after real data run)* |
| Best decision threshold | *(fill after real data run)* |

**Key SHAP findings:**
- `Tenure` — customers with short tenure churn most; loyalty programmes targeting the first 3 months could reduce churn significantly.
- `Complain` — a complaint is the single strongest churn predictor; fast complaint resolution is high ROI.
- `CashbackAmount` — higher cashback correlates with retention; consider targeted cashback for at-risk segments.

**Recommended action:** flag any customer with predicted churn probability ≥ best threshold for a proactive retention offer.